# RUKOPYS Qwen3-VL Stage 2C: Formula/Table Fine-tune - Colab A100

This notebook is shared by both specialist runs. Set `RUN_TASK = "formula"` or `RUN_TASK = "table"`, run the full notebook, save the task-specific adapter, then restart/clear VRAM before running the other task.

Canonical Drive layout:

```text
/content/drive/MyDrive/rukopys/
  data/      # required crop dataset zip
  model/     # optional local base model mirror
  adapter/   # required starting LoRA adapter for Stage 2C
  resume/    # optional task-specific checkpoint-* folders: resume/formula or resume/table
  output/    # task-specific training checkpoints and final adapters
```

Expected task outputs:

```text
formula -> output/stage2c_formula_aug and output/qwen3vl_rukopys_stage2c_formula_aug_lora_final
table   -> output/stage2c_table_aug and output/qwen3vl_rukopys_stage2c_table_aug_lora_final
```

The crop zip is copied from Drive to `/content/rukopys/data`, extracted on local Colab disk, and training reads the extracted local files for faster I/O.

Expected extracted crop layout:

```text
/content/rukopys/data/ocr_region_crops_wrong_aug/
  labels.jsonl
  formula/*.jpg
  table/*.jpg
  stats.json
```


In [ ]:
# Colab dependency + Drive setup cell.
INSTALL_DEPS = True
MOUNT_DRIVE = True

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

if INSTALL_DEPS:
    import subprocess
    import sys

    commands = [
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade-strategy",
            "only-if-needed",
            "accelerate",
            "peft",
            "bitsandbytes",
            "trl",
            "qwen-vl-utils",
            "datasets",
            "pandas==2.2.2",
            "pillow<12",
        ],
        [sys.executable, "-m", "pip", "install", "-q", "-U", "git+https://github.com/huggingface/transformers.git"],
    ]
    for cmd in commands:
        print("Running:", " ".join(cmd), flush=True)
        subprocess.check_call(cmd)

# Put the crop dataset zip on Drive, for example:
# /content/drive/MyDrive/rukopys/data/ocr-region-crops-wrong-aug.zip
# The next config cell copies it to /content/rukopys/data and extracts it on local Colab disk.


In [ ]:
import gc
import json
import os
import random
import shutil
import time
import zipfile
from collections import Counter
from pathlib import Path

import torch
from datasets import Dataset

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

# Colab/Drive layout
LOCAL_ROOT = Path("/content/rukopys")
DRIVE_ROOT = Path("/content/drive/MyDrive/rukopys")
LOCAL_DATA_ROOT = LOCAL_ROOT / "data"
DRIVE_DATA_ROOT = DRIVE_ROOT / "data"
DRIVE_RESUME_DIR = DRIVE_ROOT / "resume"
OUTPUT_DIR = DRIVE_ROOT / "output"

LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_DATA_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_RESUME_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Fixed canonical paths. Edit these four values if your Drive layout is different.
BASE_MODEL_ID_OR_PATH = DRIVE_ROOT / "model/qwen3vl-8b-instruct"
START_LORA_DIR = DRIVE_ROOT / "adapter/qwen3vl_rukopys_lora_final"
DRIVE_OCR_CROPS_ZIP = DRIVE_DATA_ROOT / "ocr-region-crops-wrong-aug.zip"
OCR_CROPS_ROOT = LOCAL_DATA_ROOT / "ocr_region_crops_wrong_aug"

START_LORA_DIR.mkdir(parents=True, exist_ok=True)

# Crop dataset zip lives on Drive, but training reads extracted files from local Colab disk.
PREPARE_OCR_CROPS_FROM_DRIVE_ZIP = True
FORCE_RECOPY_OCR_CROPS_ZIP = False
FORCE_REEXTRACT_OCR_CROPS_ZIP = False
def prepare_ocr_crops_from_drive_zip():
    if not PREPARE_OCR_CROPS_FROM_DRIVE_ZIP:
        return
    labels_path = OCR_CROPS_ROOT / "labels.jsonl"
    if labels_path.exists() and not FORCE_REEXTRACT_OCR_CROPS_ZIP:
        print("Using existing local extracted OCR crops:", OCR_CROPS_ROOT)
        return

    source_zip = DRIVE_OCR_CROPS_ZIP
    if not source_zip.exists():
        raise FileNotFoundError(f"No crop dataset zip found at {source_zip}")
    local_zip = LOCAL_DATA_ROOT / source_zip.name
    needs_copy = (
        FORCE_RECOPY_OCR_CROPS_ZIP
        or not local_zip.exists()
        or local_zip.stat().st_size != source_zip.stat().st_size
    )
    if needs_copy:
        print(f"Copying crop dataset zip to local disk: {source_zip} -> {local_zip}", flush=True)
        shutil.copy2(source_zip, local_zip)
    else:
        print("Using existing local crop dataset zip:", local_zip)

    print(f"Extracting crop dataset zip under {LOCAL_DATA_ROOT}", flush=True)
    with zipfile.ZipFile(local_zip) as zf:
        zf.extractall(LOCAL_DATA_ROOT)

    if not labels_path.exists():
        raise FileNotFoundError(
            f"Copied and extracted the crop zip, but no labels.jsonl was found at {labels_path}. "
            "Update OCR_CROPS_ROOT if the zip extracts to a different folder."
        )
    print("OCR crops ready on local disk:", OCR_CROPS_ROOT)


prepare_ocr_crops_from_drive_zip()

# Set one specialist run at a time: "formula" or "table".
RUN_TASK = "table"
VALID_RUN_TASKS = {"formula", "table"}
if RUN_TASK not in VALID_RUN_TASKS:
    raise ValueError(f"RUN_TASK must be one of {sorted(VALID_RUN_TASKS)}, got {RUN_TASK!r}")

RUN_NAME = f"stage2c_{RUN_TASK}_aug"
TARGET_TYPES = {RUN_TASK}
TRAIN_OUTPUT_DIR = OUTPUT_DIR / RUN_NAME
FINAL_ADAPTER_DIR = OUTPUT_DIR / f"qwen3vl_rukopys_{RUN_NAME}_lora_final"
RUN_RESUME_DIR = DRIVE_RESUME_DIR / RUN_TASK
CALLBACK_NAME = f"stage2c-{RUN_TASK}-aug"
RUN_RESUME_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_TRAIN_SAMPLES = None  # Optional smoke limit; keep None for the full task-specific dataset.
VAL_RATIO = 0.15
EARLY_STOPPING_PATIENCE = 5
EARLY_STOPPING_THRESHOLD = 0.0

NUM_TRAIN_EPOCHS = 3
PER_DEVICE_BATCH = 16
GRAD_ACCUM = 16
DATALOADER_NUM_WORKERS = 2
MODEL_DEVICE_MAP = "auto"
MAX_SEQ_LENGTH = 3072
MAX_PIXELS_CROP = 320_000

PRINT_STEPS = 5
SAVE_STEPS = 20
EVAL_STEPS = 10
SAVE_TOTAL_LIMIT = 4
RESUME_TRAINING = True
RESUME_CHECKPOINT_DIR = str(RUN_RESUME_DIR)  # Optional task-specific checkpoint-* folder root. Empty is OK.


In [ ]:
SOURCE_HINTS = {
    "dictation": "Ukrainian dictation handwriting. Do not complete from canonical text; read only visible characters.",
    "archive": "Historical Ukrainian/Cyrillic document. Preserve old spelling; do not modernize.",
    "school": "School homework. It may contain corrections, teacher marks, formulas, and mixed handwriting/print.",
    "university": "University exam/coursework. It may contain formulas, tables, chemistry notation, and technical symbols.",
}
DEFAULT_SOURCE_HINT = "Read only visible characters from this crop."

SPECIAL_TEXT_MARKER_RULES = (
    "Use [illegible] only for unreadable words inside an otherwise legible text region. "
    "Use ~~word~~ for visible strikethrough and ~~old~~{new} for visible correction."
)

STAGE_B_GUARDRAILS = (
    "The final transcription must be supported by the crop. "
    "Do not complete missing words from source hint, language prior, or canonical dictation text. "
    "Do not translate, correct grammar, normalize spelling, expand abbreviations, summarize, "
    "or infer hidden/missing text. No JSON, no Markdown, no explanation."
)

CROP_PROMPTS = {
    "formula": (
        "Read this standalone math, logic, vector, matrix, determinant, set/relation, statistics, physics, "
        "or chemistry expression exactly as written. Return only formula text, using LaTeX when it is the "
        "clearest representation and plain Unicode when it better matches the handwriting. Preserve visible "
        "symbols, indices, superscripts, subscripts, arrows, fractions, matrix/determinant structure, punctuation, "
        "numbering, and strikethrough/correction markers. Do not solve, simplify, normalize, explain, or convert "
        "old notation into a different style."
    ),
    "table": (
        "Read this table region exactly. Return only pipe-separated table text. Use one output line per visual row "
        "and | between cells. Preserve empty cells with empty fields, for example A||C. Preserve row order, "
        "column order, multi-word cell text, wrapped cell text, numbers, units, punctuation, dashes, and visible "
        "spelling mistakes. Do not infer missing cells, do not rebalance columns, do not summarize, and do not explain."
    ),
}


def normalize_source(value):
    value = str(value or "").strip().lower()
    return value if value in SOURCE_HINTS else "default"


def build_crop_prompt(region_type, source=None):
    source_hint = SOURCE_HINTS.get(normalize_source(source), DEFAULT_SOURCE_HINT)
    type_prompt = CROP_PROMPTS[region_type]
    return "\n".join([source_hint, type_prompt, SPECIAL_TEXT_MARKER_RULES, STAGE_B_GUARDRAILS])


def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def resolve_cached_image_path(data_root, row):
    raw = Path(str(row.get("crop_path") or row.get("image_path") or row.get("relative_image_path") or ""))
    if not raw.is_absolute() and raw.parts and raw.parts[0] == data_root.name:
        raw = Path(*raw.parts[1:])
    image_path = raw if raw.is_absolute() else data_root / raw
    if not image_path.exists():
        raise FileNotFoundError(f"Cached crop not found at {image_path}")
    return str(image_path)


def label_to_sample(data_root, row):
    region_type = str(row.get("type") or "").strip().lower()
    if region_type not in TARGET_TYPES:
        return None
    answer = "" if row.get("text") is None else str(row.get("text"))
    if not answer.strip():
        return None
    return {
        "image_path": resolve_cached_image_path(data_root, row),
        "prompt": build_crop_prompt(region_type, source=row.get("source")),
        "answer": answer,
        "region_type": region_type,
        "source": row.get("source", "unknown"),
        "augment": bool(row.get("augment")),
        "aug_id": int(row.get("aug_id") or 0),
        "source_kind": row.get("source_kind", "unknown"),
        "crop_path": row.get("crop_path", ""),
    }


model_id = str(BASE_MODEL_ID_OR_PATH)
if model_id.startswith("/") and not Path(model_id).exists():
    raise FileNotFoundError(f"BASE_MODEL_ID_OR_PATH does not exist: {model_id}")

start_lora_dir = Path(START_LORA_DIR)
if not (start_lora_dir / "adapter_config.json").exists():
    raise FileNotFoundError(f"No adapter_config.json found in START_LORA_DIR: {start_lora_dir}")

ocr_crops_root = Path(OCR_CROPS_ROOT)
labels_path = ocr_crops_root / "labels.jsonl"
if not labels_path.exists():
    raise FileNotFoundError(f"No labels.jsonl found in OCR_CROPS_ROOT: {ocr_crops_root}")
stats_path = ocr_crops_root / "stats.json"

labels = read_jsonl(labels_path)
samples = [label_to_sample(ocr_crops_root, row) for row in labels]
samples = [row for row in samples if row is not None]
random.Random(SEED).shuffle(samples)
if MAX_TRAIN_SAMPLES is not None:
    samples = samples[:MAX_TRAIN_SAMPLES]

if not samples:
    raise RuntimeError(f"No training samples found for RUN_TASK={RUN_TASK!r} in {labels_path}")

if len(samples) < 2:
    raise RuntimeError(f"Need at least 2 samples to create a validation split for RUN_TASK={RUN_TASK!r}")

val_size = max(1, round(len(samples) * VAL_RATIO))
val_size = min(val_size, len(samples) - 1)
val_samples = samples[:val_size]
train_samples = samples[val_size:]

train_ds = Dataset.from_list(train_samples)
eval_ds = Dataset.from_list(val_samples)
sample_counts_by_type = dict(Counter(row["region_type"] for row in samples))
train_counts_by_type = dict(Counter(row["region_type"] for row in train_samples))
val_counts_by_type = dict(Counter(row["region_type"] for row in val_samples))
augmented_counts_by_type = dict(Counter(row["region_type"] for row in samples if row.get("augment")))
train_augmented_counts_by_type = dict(Counter(row["region_type"] for row in train_samples if row.get("augment")))
val_augmented_counts_by_type = dict(Counter(row["region_type"] for row in val_samples if row.get("augment")))
source_kind_counts = dict(Counter(row.get("source_kind", "unknown") for row in samples))
prompt_config = {
    "stage": f"{RUN_NAME}_dataset",
    "run_task": RUN_TASK,
    "target_types": sorted(TARGET_TYPES),
    "training_output_dir": str(TRAIN_OUTPUT_DIR),
    "resume_dir": str(RUN_RESUME_DIR),
    "final_adapter_dir": str(FINAL_ADAPTER_DIR),
    "sample_counts_by_type": sample_counts_by_type,
    "train_counts_by_type": train_counts_by_type,
    "val_counts_by_type": val_counts_by_type,
    "augmented_counts_by_type": augmented_counts_by_type,
    "train_augmented_counts_by_type": train_augmented_counts_by_type,
    "val_augmented_counts_by_type": val_augmented_counts_by_type,
    "source_kind_counts": source_kind_counts,
    "val_ratio": VAL_RATIO,
    "train_samples": len(train_samples),
    "val_samples": len(val_samples),
    "source_hints": SOURCE_HINTS,
    "default_source_hint": DEFAULT_SOURCE_HINT,
    "crop_prompts": CROP_PROMPTS,
    "special_text_marker_rules": SPECIAL_TEXT_MARKER_RULES,
    "stage_b_guardrails": STAGE_B_GUARDRAILS,
    "crop_prompt_assembly": "source_hint + type_prompt + special_text_marker_rules + stage_b_guardrails",
}
if stats_path.exists():
    prompt_config["data_stats"] = json.loads(stats_path.read_text(encoding="utf-8"))

print("RUN_TASK:", RUN_TASK)
print("Base model:", model_id)
print("Starting LoRA:", start_lora_dir)
print("OCR crops root:", ocr_crops_root)
print("Training output:", TRAIN_OUTPUT_DIR)
print("Final adapter:", FINAL_ADAPTER_DIR)
print("Resume dir:", RUN_RESUME_DIR)
print("Training samples:", len(samples))
print("Train samples:", len(train_samples))
print("Validation samples:", len(val_samples))
print("Counts by type:", sample_counts_by_type)
print("Train counts by type:", train_counts_by_type)
print("Validation counts by type:", val_counts_by_type)
print("Augmented by type:", augmented_counts_by_type)
print("Train augmented by type:", train_augmented_counts_by_type)
print("Validation augmented by type:", val_augmented_counts_by_type)
print("Source kind counts:", source_kind_counts)


In [ ]:

from peft import PeftModel, prepare_model_for_kbit_training
from qwen_vl_utils import process_vision_info
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig, EarlyStoppingCallback, TrainerCallback
from trl import SFTConfig, SFTTrainer


def force_bf16_config(model):
    model.config.torch_dtype = torch.bfloat16
    for attr in ("text_config", "vision_config"):
        cfg = getattr(model.config, attr, None)
        if cfg is not None:
            cfg.torch_dtype = torch.bfloat16
            if hasattr(cfg, "dtype"):
                cfg.dtype = "bfloat16"


processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

base_model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    device_map=MODEL_DEVICE_MAP,
    quantization_config=quantization_config,
    dtype=torch.bfloat16,
    trust_remote_code=True,
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
)
force_bf16_config(base_model)
base_model.config.use_cache = False
base_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=True)
try:
    base_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
except TypeError:
    base_model.gradient_checkpointing_enable()

model = PeftModel.from_pretrained(base_model, str(start_lora_dir), is_trainable=True)
for _, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)
model.print_trainable_parameters()


In [ ]:

def build_messages(sample):
    return [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": sample["image_path"], "max_pixels": MAX_PIXELS_CROP},
                {"type": "text", "text": sample["prompt"]},
            ],
        },
        {"role": "assistant", "content": [{"type": "text", "text": sample["answer"]}]},
    ]


def encode_marker(tokenizer):
    try:
        return tokenizer.encode("<|im_start|>assistant\n", allowed_special="all", add_special_tokens=False)
    except TypeError:
        return tokenizer.encode("<|im_start|>assistant\n", add_special_tokens=False)


ASSISTANT_MARKER = encode_marker(processor.tokenizer)


def data_collator(examples):
    messages_list = [build_messages(ex) for ex in examples]
    texts = [
        processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=False)
        for msg in messages_list
    ]
    image_inputs, video_inputs = process_vision_info(messages_list)
    batch = processor(
        text=texts,
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        return_tensors="pt",
    )

    labels = batch["input_ids"].clone()
    pad_id = processor.tokenizer.pad_token_id
    if pad_id is not None:
        labels[labels == pad_id] = -100

    eos_id = processor.tokenizer.eos_token_id
    for i in range(labels.shape[0]):
        ids = batch["input_ids"][i].tolist()
        start = -1
        for j in range(0, len(ids) - len(ASSISTANT_MARKER) + 1):
            if ids[j:j + len(ASSISTANT_MARKER)] == ASSISTANT_MARKER:
                start = j + len(ASSISTANT_MARKER)
                break
        actual_len = int(batch["attention_mask"][i].sum().item())
        truncated = actual_len >= MAX_SEQ_LENGTH and (eos_id is None or ids[actual_len - 1] != eos_id)
        if start >= 0 and not truncated:
            labels[i, :start] = -100
        else:
            labels[i, :] = -100

    batch["labels"] = labels
    for key, value in list(batch.items()):
        if isinstance(value, torch.Tensor) and value.dtype == torch.float32:
            batch[key] = value.to(torch.bfloat16)
    return batch


In [ ]:

class PrintProgressCallback(TrainerCallback):
    def __init__(self, name):
        self.name = name
        self.start_time = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.start_time = time.time()
        print(f"[{self.name}] start: max_steps={state.max_steps}, grad_accum={args.gradient_accumulation_steps}", flush=True)

    def on_log(self, args, state, control, logs=None, **kwargs):
        logs = logs or {}
        elapsed = time.time() - (self.start_time or time.time())
        step = max(1, state.global_step)
        eta = (elapsed / step) * max(0, state.max_steps - step)
        loss = logs.get("loss", logs.get("eval_loss", None))
        lr = logs.get("learning_rate", None)
        msg = f"[{self.name}] step {state.global_step}/{state.max_steps}"
        if loss is not None:
            msg += f" loss={loss:.4f}"
        if lr is not None:
            msg += f" lr={lr:.2e}"
        msg += f" elapsed={elapsed/60:.1f}m eta={eta/60:.1f}m"
        if torch.cuda.is_available():
            msg += " vram=" + ",".join(
                f"{i}:{torch.cuda.memory_allocated(i)/1024**3:.1f}GB"
                for i in range(torch.cuda.device_count())
            )
        print(msg, flush=True)

    def on_step_end(self, args, state, control, **kwargs):
        step = int(state.global_step or 0)
        if step <= 0 or step % PRINT_STEPS != 0:
            return
        elapsed = time.time() - (self.start_time or time.time())
        eta = (elapsed / max(1, step)) * max(0, state.max_steps - step)
        msg = f"[{self.name}] print step {step}/{state.max_steps} elapsed={elapsed/60:.1f}m eta={eta/60:.1f}m"
        if torch.cuda.is_available():
            msg += " vram=" + ",".join(
                f"{i}:{torch.cuda.memory_allocated(i)/1024**3:.1f}GB"
                for i in range(torch.cuda.device_count())
            )
        print(msg, flush=True)


class OOMRecoverySFTTrainer(SFTTrainer):
    def training_step(self, model, inputs, num_items_in_batch=None):
        try:
            try:
                loss = super().training_step(model, inputs, num_items_in_batch=num_items_in_batch)
            except TypeError:
                loss = super().training_step(model, inputs)
            if self.args.device != loss.device:
                loss = loss.to(self.args.device)
            return loss
        except torch.cuda.OutOfMemoryError:
            print("OOM: skipping one batch after clearing cache.", flush=True)
            for p in model.parameters():
                p.grad = None
            torch.cuda.empty_cache()
            gc.collect()
            return torch.tensor(0.0, device=self.args.device)


def find_latest_checkpoint(root):
    root = Path(root)
    if not root.exists():
        return None
    checkpoints = []
    for path in root.glob("checkpoint-*"):
        try:
            step = int(path.name.rsplit("-", 1)[-1])
        except ValueError:
            continue
        if (path / "trainer_state.json").exists():
            checkpoints.append((step, path))
    if not checkpoints:
        return None
    return str(max(checkpoints, key=lambda item: item[0])[1])


def resolve_resume_checkpoint(output_dir):
    if not RESUME_TRAINING:
        return None
    search_roots = []
    for item in (RESUME_CHECKPOINT_DIR, output_dir):
        if not item:
            continue
        path = Path(item)
        if path not in search_roots:
            search_roots.append(path)
    for resume_root in search_roots:
        if not resume_root.exists():
            continue
        if (resume_root / "trainer_state.json").exists():
            return str(resume_root)
        latest = find_latest_checkpoint(resume_root)
        if latest:
            return latest
    return None


sft_config_kwargs = dict(
    output_dir=str(TRAIN_OUTPUT_DIR),
    per_device_train_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=8e-6,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    fp16=False,
    bf16=True,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    max_grad_norm=0.3,
    logging_steps=10,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    eval_steps=EVAL_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    remove_unused_columns=False,
    gradient_checkpointing=True,
    dataloader_num_workers=DATALOADER_NUM_WORKERS,
    dataloader_pin_memory=False,
    dataloader_persistent_workers=DATALOADER_NUM_WORKERS > 0,
    dataset_text_field="",
    dataset_kwargs={"skip_prepare_dataset": True},
)
try:
    training_args = SFTConfig(eval_strategy="steps", **sft_config_kwargs)
except TypeError:
    training_args = SFTConfig(evaluation_strategy="steps", **sft_config_kwargs)

trainer = OOMRecoverySFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
    callbacks=[
        PrintProgressCallback(CALLBACK_NAME),
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            early_stopping_threshold=EARLY_STOPPING_THRESHOLD,
        ),
    ],
)
resume_checkpoint = resolve_resume_checkpoint(training_args.output_dir)
if resume_checkpoint:
    print("Resuming from checkpoint:", resume_checkpoint, flush=True)
else:
    print(f"No resume checkpoint found; starting a fresh Stage 2C {RUN_TASK} run.", flush=True)
trainer.train(resume_from_checkpoint=resume_checkpoint)

final_dir = FINAL_ADAPTER_DIR
trainer.model.save_pretrained(final_dir)
processor.save_pretrained(final_dir)

prompt_config.update({
    "stage": f"{RUN_NAME}_cached_finetune",
    "run_task": RUN_TASK,
    "target_types": sorted(TARGET_TYPES),
    "training_output_dir": str(TRAIN_OUTPUT_DIR),
    "resume_dir": str(RUN_RESUME_DIR),
    "final_adapter_dir": str(FINAL_ADAPTER_DIR),
    "learning_rate": 8e-6,
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "val_ratio": VAL_RATIO,
    "train_samples": len(train_samples),
    "val_samples": len(val_samples),
    "evaluation_strategy": "steps",
    "eval_steps": EVAL_STEPS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "early_stopping_threshold": EARLY_STOPPING_THRESHOLD,
    "metric_for_best_model": "eval_loss",
    "greater_is_better": False,
    "best_model_checkpoint": trainer.state.best_model_checkpoint,
    "best_metric": trainer.state.best_metric,
    "save_steps": SAVE_STEPS,
    "save_total_limit": SAVE_TOTAL_LIMIT,
    "resume_from_checkpoint": resume_checkpoint,
    "base_model": str(model_id),
    "start_lora": str(start_lora_dir),
    "ocr_crops_root": str(ocr_crops_root),
    "labels_path": str(labels_path),
})
(final_dir / "rukopys_prompt_config.json").write_text(
    json.dumps(prompt_config, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(f"Saved final Stage 2C {RUN_TASK} LoRA adapter to", final_dir)
